In [ ]:
import os

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.optim as optim

import optuna

import numpy as np

import albumentations as A

import SimpleITK
import SimpleITK as sitk

from typing import Any, Dict, Tuple
import re
import random

In [13]:
BASE_DIR = "database/"  # Name of de dataset folder

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():  # mac
    device = torch.device("mps")
else:
    device = torch.device("cpu")


print("Using device:", device)

sitk.ProcessObject_SetGlobalWarningDisplay(False)

Using device: mps


## 1. FCN1:


In [14]:
def get_file_paths(directory):
    """
    Get all file paths in a directory and its subdirectories.
    """
    file_paths = []  # creates an empty list
    for root, dirs, files in os.walk(directory):
        for file in files:
            file_paths.append(os.path.join(root, file))
    return file_paths


resized = A.Compose(
    [A.Resize(height=256, width=256)]
)  # re-sized all the images to 256 × 256 pixels


### 1.1 FCN1 - Dataset


In [15]:
class FCN1Dataset(Dataset):
    def __init__(self, filepath, transform=None):
        self.slices = []
        self.transform = transform

        for path in filepath:
            img = self._sitk_load(path)

            mask_path = path.replace(".nii.gz", "_gt.nii.gz")
            mask = self._sitk_load(mask_path)
            mask = (mask == 3).astype(np.float32)

            # regex to find info
            patient = path.split("/")[-2]
            frame = int(re.search(r"frame(\d+)", path).group(1))

            for i in range(img.shape[0]):
                self.slices.append(
                    {
                        "image": img[i],
                        "mask": mask[i],
                        "patient": patient,
                        "frame": frame,
                        "slice_idx": i,
                    }
                )

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        item = self.slices[idx]
        img = item["image"]
        mask = item["mask"]

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented["image"]
            mask = augmented["mask"]

        img = torch.from_numpy(img).float()
        mask = torch.from_numpy(mask).float()

        img = img.unsqueeze(0)
        mask = mask.unsqueeze(0)

        metadata = {
            "patient": item["patient"],
            "frame": item["frame"],
            "slice_idx": item["slice_idx"],
        }

        return img, mask, metadata

    def _sitk_load(
        self,
        filepath,
    ) -> Tuple[np.ndarray, Dict[str, Any]]:
        """Loads an image using SimpleITK and returns the image and its metadata.
        Args:
            filepath: Path to the image.

        Returns:
            - ([N], H, W), Image array.

        """
        # Load image and save info
        image = SimpleITK.ReadImage(str(filepath))
        # Extract numpy array from the SimpleITK image object
        im_array = np.squeeze(SimpleITK.GetArrayFromImage(image))

        return im_array


In [16]:
all_files = get_file_paths("database/")

img_paths = [
    file
    for file in all_files
    if file.endswith(".nii.gz") and "_gt" not in file and "_4d" not in file
]
train_val_paths = [file for file in img_paths if "training" in file]


# Choose 10 random patients from pattient 001 to patiento 100 for the validation set
random.seed(42)
patients = list(range(1, 101))  # patient001 - patient100
val_patients = random.sample(patients, 10)

train_paths = []
validation_paths = []

# Extracts the patient number from the file path and checks whether that patient belongs to the validation set.
for f in train_val_paths:
    patient = int(f.split("patient")[1][:3])
    if patient in val_patients:
        validation_paths.append(f)
    else:
        train_paths.append(f)

# ------

ds_train = FCN1Dataset(train_paths, resized)
ds_validation = FCN1Dataset(validation_paths, resized)

test_image_paths = [file for file in img_paths if "testing" in file]
test_image_sorted = sorted(
    test_image_paths, key=lambda x: int(re.search(r"patient(\d+)", x).group(1))
)
ds_test = FCN1Dataset(test_image_sorted, resized)

### 1.3. FCN1 - Architecture


In [17]:
class EncoderFCN1(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)

        skip = x
        x = self.pool(x)
        return x, skip


class DecoderFCN1(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        # Input: 512 - Output: 256
        self.upconv = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        # Input: 256 *2 -> (because of skip connections)
        self.conv1 = nn.Conv2d(out_channels * 2, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)

        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

    def forward(self, x, skip):
        x = self.upconv(x)
        # dim = 1: Concatenate along the channel dimension.
        x = torch.cat((x, skip), dim=1)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        return x


class FCN1Net(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        self.encoder1 = EncoderFCN1(1, 32)
        self.encoder2 = EncoderFCN1(32, 64)
        self.encoder3 = EncoderFCN1(64, 128)
        self.encoder4 = EncoderFCN1(128, 256)

        # Bottleneck
        self.bottleneck1 = nn.Conv2d(256, 512, 3, padding=1)
        self.bn_b1 = nn.BatchNorm2d(512)
        # bottleneck2
        self.bottleneck2 = nn.Conv2d(512, 512, 3, padding=1)
        self.bn_b2 = nn.BatchNorm2d(512)
        self.relu = nn.ReLU()

        # Decoder
        self.decode1 = DecoderFCN1(512, 256)
        self.decode2 = DecoderFCN1(256, 128)
        self.decode3 = DecoderFCN1(128, 64)
        self.decode4 = DecoderFCN1(64, 32)

        self.final_conv = nn.Conv2d(32, 1, 3, padding=1)

        # init weights AFTER all layers exist
        self.apply(self.init_weights)

    def forward(self, x):
        x, s1 = self.encoder1(x)
        x, s2 = self.encoder2(x)
        x, s3 = self.encoder3(x)
        x, s4 = self.encoder4(x)

        x = self.bottleneck1(x)
        x = self.bn_b1(x)
        x = self.relu(x)
        x = self.bottleneck2(x)
        x = self.bn_b2(x)
        x = self.relu(x)

        x = self.decode1(x, s4)
        x = self.decode2(x, s3)
        x = self.decode3(x, s2)
        x = self.decode4(x, s1)

        x = self.final_conv(x)
        return x

    def init_weights(self, m):
        if isinstance(m, nn.Conv2d) or isinstance(m, nn.ConvTranspose2d):
            init.kaiming_normal_(m.weight, nonlinearity="relu")

### 1.3. FCN1 - Training


## OPTUNA


In [ ]:
random.seed(42)
torch.manual_seed(42)
np.random.seed(42)


N_EPOCHS = 20


def objective(trial):
    torch.manual_seed(42)
    np.random.seed(42)

    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])
    lr_ = trial.suggest_categorical("lr_", [0.01, 0.001, 0.0001])

    model = FCN1Net().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr_)
    criterion = nn.BCEWithLogitsLoss()

    train_loader = DataLoader(ds_train, batch_size=batch_size, shuffle=True)

    valid_loader = DataLoader(ds_validation, batch_size=batch_size, shuffle=False)

    for epoch in range(N_EPOCHS):
        model.train()
        for images, masks, _ in train_loader:
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

    model.eval()
    val_losses = []
    with torch.no_grad():
        for images, masks, _ in valid_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            val_losses.append(loss.item())

    val_loss = float(np.mean(val_losses))

    print(
        f"Trial {trial.number}: batch_size={batch_size}, lr_={lr_:.6f}, val_loss={val_loss:.4f}"
    )

    return val_loss


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

print("\nStudy statistics:")
print("  Number of finished trials: ", len(study.trials))
print("  Best value: ", study.best_value)
print("  Best params: ")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

print("\nBest combination found by Optuna:")
print(f"  batch_size = {study.best_params['batch_size']}")
print(f"  lr_ = {study.best_params['lr_']}")

[I 2026-07-02 14:50:42,852] A new study created in memory with name: no-name-04ece24e-63ee-48da-984f-e61d7092be9e
